# A reproducible alpha research pipeline on WorldQuant BRAINThis notebook is the **method** of a research pipeline I built for myself: a way to keep alpharesearch reproducible, resumable and reviewable instead of living in a browser tab.It is deliberately a **skeleton with the strategy removed**. What it contains:- how every knob is moved out of the code and into one configuration file- how a field library is built so that coverage is not dominated by a single dataset- where a human decision belongs, and where it does not- how candidates are screened, clustered for duplication and exported- how the platform's own verdict is read instead of re-implementing its thresholdsWhat it deliberately **does not** contain: the field list, the expression grammar with myoperator and window choices, any per-candidate metrics, my submitted portfolio, or thefailure analysis. Those stay on my machine. The point of publishing the method is that thepipeline is the reusable part; the signal is the part that is not.

## 0. Configuration — every knob lives in one fileRegion, universe, delay, windows, operators, interaction pairs, thresholds and the paths ofevery artefact are read from `factory_config.json`. The notebook only injects them.Change the experiment by editing the JSON. Change the pipeline by editing a cell.

In [ ]:
import json, os, time, subprocessfrom pathlib import Pathimport numpy as npimport pandas as pdimport requestsfrom requests.auth import HTTPBasicAuthCONFIG_PATH = Path(os.environ.get("ALPHA_CONFIG", "factory_config.json"))CONFIG = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))def cfg(path, default=None):    """Read a dotted path out of the config, with a default."""    node = CONFIG    for key in path.split("."):        if not isinstance(node, dict) or key not in node:            return default        node = node[key]    return nodeREGION = cfg("universe.region", "USA")UNIVERSE = cfg("universe.name", "TOP3000")DELAY = cfg("universe.delay", 1)ARTIFACTS = Path(cfg("paths.artifacts", "artifacts"))ARTIFACTS.mkdir(parents=True, exist_ok=True)REGISTRY_PATH = ARTIFACTS / cfg("paths.registry", "alpha_registry.csv")REVIEW_PATH = ARTIFACTS / cfg("paths.review", "field_review.csv")CONFIG_EXAMPLE = Path("factory_config.example.json")print(f"region={REGION} universe={UNIVERSE} delay={DELAY}")print(f"registry -> {REGISTRY_PATH}")

## 1. Credentials and a single API clientCredentials are read from the environment, never from the notebook and never from a filethat could be committed. Every platform call goes through one client so that rate limiting,retry and error handling exist in exactly one place.Two habits matter here and are the reason this cell is short:1. **A shared request gate.** The platform throttles. All research processes take the same   lock file, keep a minimum interval between requests and back off using the retry hint   when they hit a limit. After an account is throttled, nothing works for a while, so this   is a requirement, not an optimisation.2. **Resume instead of restart.** Every result is written to disk as it arrives, and a run   skips expressions that are already recorded.

In [ ]:
MIN_REQUEST_INTERVAL = cfg("api.min_interval_seconds", 2.0)REQUEST_LOCK = Path(cfg("api.lock_file", ".alpha_api.lock"))USERNAME_ENV = ["WORLDQUANT_USERNAME", "BRAIN_USERNAME"]PASSWORD_ENV = ["WORLDQUANT_PASSWORD", "BRAIN_PASSWORD"]BASE_URL = cfg("api.base_url", "https://api.worldquantbrain.com")def _credentials():    for user_key, pass_key in zip(USERNAME_ENV, PASSWORD_ENV):        user, password = os.environ.get(user_key), os.environ.get(pass_key)        if user and password:            return user, password    raise RuntimeError(        "Set the platform credentials in the environment "        f"({USERNAME_ENV[0]}/{PASSWORD_ENV[0]} or {USERNAME_ENV[1]}/{PASSWORD_ENV[1]})."    )class RequestGate:    """One lock, one minimum interval, retry-aware backoff."""    def __init__(self, path, interval):        self.path, self.interval, self.last = Path(path), interval, 0.0    def wait(self):        elapsed = time.time() - self.last        if elapsed < self.interval:            time.sleep(self.interval - elapsed)        self.last = time.time()class BrainClient:    def __init__(self, gate):        self.gate = gate        user, password = _credentials()        self.session = requests.Session()        self.session.auth = HTTPBasicAuth(user, password)        self.session.headers.update({"Content-Type": "application/json"})    def _request(self, method, path, **kwargs):        self.gate.wait()        response = self.session.request(method, f"{BASE_URL}{path}", timeout=90, **kwargs)        if response.status_code == 429:            retry_after = float(response.headers.get("Retry-After", self.gate.interval * 5))            time.sleep(retry_after)            return self._request(method, path, **kwargs)        response.raise_for_status()        return response    # --- platform surface used by the pipeline -------------------------    def get_datasets(self, region, universe, delay):        r = self._request("GET", "/datasets",                          params={"instrumentType": "EQUITY", "region": region,                                  "universe": universe, "delay": delay})        return r.json().get("results", [])    def get_fields(self, dataset_id, universe, delay, limit):        r = self._request("GET", "/data-fields",                          params={"dataset.id": dataset_id, "universe": universe,                                  "delay": delay, "limit": limit})        return r.json().get("results", [])    def simulate(self, expression, settings):        r = self._request("POST", "/simulations",                          json={"type": "REGULAR", "regular": expression, "settings": settings})        return r.headers.get("Location")    def poll(self, location, timeout=300):        deadline = time.time() + timeout        while time.time() < deadline:            r = self._request("GET", location)            payload = r.json()            if payload.get("status") in {"COMPLETE", "ERROR"}:                return payload            time.sleep(2)        raise TimeoutError("simulation did not finish in time")    def get_alpha(self, alpha_id):        return self._request("GET", f"/alphas/{alpha_id}").json()GATE = RequestGate(REQUEST_LOCK, MIN_REQUEST_INTERVAL)client = BrainClient(GATE)print("client ready; credentials loaded from the environment")

## 2. Discover datasetsThe available datasets depend on the region, universe and delay. Changing any of them meansre-running this cell, not editing the pipeline.

In [ ]:
datasets = client.get_datasets(REGION, UNIVERSE, DELAY)datasets_df = pd.DataFrame([    {"id": d.get("id"), "name": d.get("name"), "category": d.get("category"),     "coverage": d.get("coverage"), "valueScore": d.get("valueScore")}    for d in datasets])print(f"datasets discovered: {len(datasets_df)}")datasets_df.head(10)

## 3. Build the field libraryThis step is the foundation of the whole pipeline. The first version of this code had a bug:it walked the datasets, accumulated fields until a limit, and stopped. Every field it collectedcame from a single dataset, so the "field library" had no diversity and no combination searchover it could find anything.The fix is small and worth stating, because the same mistake reappears in everyfield-collection loop: **take a bounded number of fields from each dataset**, so the librarycovers the space instead of one corner of it.

In [ ]:
MAX_FIELDS_PER_DATASET = cfg("fields.max_per_dataset", 60)DATASET_LIMIT = cfg("fields.dataset_limit", 12)fields = []for dataset in datasets[:DATASET_LIMIT]:    try:        fetched = client.get_fields(dataset["id"], UNIVERSE, DELAY, MAX_FIELDS_PER_DATASET)    except Exception as exc:                      # one dataset failing must not stop the run        print(f"skip {dataset.get('name')}: {exc}")        continue    for field in fetched:        fields.append({            "id": field.get("id"),            "dataset": dataset.get("name"),            "dataset_id": dataset.get("id"),            "type": field.get("type"),            "coverage": field.get("coverage"),            "category": field.get("category"),            "description": field.get("description"),        })fields_df = pd.DataFrame(fields).drop_duplicates(subset=["id"])print(f"field library: {len(fields_df)} fields across {fields_df['dataset'].nunique()} datasets")fields_df.groupby("dataset").size().sort_values(ascending=False).head(10)

## 4. Profile the library, then shrink itThe platform reports a coverage figure per field. Fields below a coverage floor are rarelyworth the compute, so filtering first turns "test thousands of fields" into "test a fewhundred fields that actually have data".The order matters: rotate across datasets **before** truncating, otherwise the truncationre-creates the single-dataset bug from the previous step.

In [ ]:
MIN_COVERAGE = cfg("fields.min_coverage", 0.25)MAX_FIELDS_FOR_GEN = cfg("fields.max_for_generation", 300)if "coverage" not in fields_df.columns:    fields_df["coverage"] = np.nanfields_df["coverage"] = pd.to_numeric(fields_df["coverage"], errors="coerce").fillna(0.0)profiled = fields_df[fields_df["coverage"] >= MIN_COVERAGE].copy()# rotate by dataset before truncating, so the kept set is spread across datasetsprofiled = profiled.groupby("dataset", group_keys=False).head(    max(1, MAX_FIELDS_FOR_GEN // max(profiled["dataset"].nunique(), 1)))fields_filtered = profiled.reset_index(drop=True)print(f"kept {len(fields_filtered)} of {len(fields_df)} fields "      f"(coverage >= {MIN_COVERAGE}) across {fields_filtered['dataset'].nunique()} datasets")fields_filtered["type"].value_counts().head()

## 5. Tag fields by economic meaningA field library with no semantics is a combinatorial explosion waiting to happen: the morefields you allow, the more meaningless combinations you test.The platform already publishes a category per field (price/volume, fundamental, analyst,model, macro). Map that onto an economic label, and let the mapping be extended by hand —the extension is the part that needs domain knowledge, the mapping is the part that shouldbe mechanical.

In [ ]:
CATEGORY_MAP = {    "pv":         "price",       # price and volume    "fundamental":"fundamental",  # company financials    "analyst":    "analyst",     # analyst estimates    "model":      "model",       # model-derived factors    "macro":      "macro",       # macroeconomic series}fields_filtered["economic_category"] = fields_filtered["category"].map(CATEGORY_MAP).fillna("other")print(fields_filtered["economic_category"].value_counts())

## 6. The human gateThe machine can build the library and filter it by coverage. It cannot decide which fieldsare worth testing and in which direction — that is a judgement about economics, and pretendingotherwise is how a pipeline ends up spending its compute budget on noise.So the pipeline exports a review sheet and stops. The sheet carries the fields, the coverage,the category and an approval column plus a direction column. Approving a row is the decision;everything after this cell is mechanical.For a library of a few hundred fields, reading every row is not realistic. A short list canbe approved in bulk instead — the shortlist is still a human decision, just a smaller one.

In [ ]:
APPROVED_FIELDS = cfg("review.approved_fields_file", "field_review.csv")DIRECTIONS = cfg("review.directions", {})fields_filtered["approved"] = Falsefields_filtered["direction"] = fields_filtered["economic_category"].map(DIRECTIONS).fillna(1.0)if REVIEW_PATH.exists():    review = pd.read_csv(REVIEW_PATH)    approved_ids = set(review.loc[review.get("approved", False).astype(bool), "id"].astype(str))    fields_filtered.loc[fields_filtered["id"].astype(str).isin(approved_ids), "approved"] = Truefields_filtered.to_csv(REVIEW_PATH, index=False)approved = fields_filtered[fields_filtered["approved"]]print(f"review sheet -> {REVIEW_PATH}")print(f"approved fields: {len(approved)} of {len(fields_filtered)}")approved.head(10)

## 7. Candidate generationCandidates are generated deterministically from the approved fields, using the operator andwindow lists from the configuration file. Nothing here is random, so the same configurationalways produces the same candidate set — which is what makes the experiment log meaningfullater.The generation rules themselves are a personal choice, so the configuration file ships withempty operator lists and this cell shows the shape rather than my settings.

In [ ]:
UNARY_OPERATORS = cfg("generation.unary_operators", [])TIME_OPERATORS = cfg("generation.time_operators", [])WINDOWS = cfg("generation.windows", [])INTERACTION_PAIRS = cfg("generation.interaction_pairs", [])def build_candidates(rows):    """Deterministic candidate construction from approved fields."""    candidates = []    for row in rows:        field_id, category = row["id"], row["economic_category"]        direction = row.get("direction", 1.0)        sign = "-" if direction < 0 else ""        for op in UNARY_OPERATORS:            candidates.append({"expression": f"{sign}rank({op}({field_id}))",                               "field_ids": field_id, "category": category})        for op in TIME_OPERATORS:            for window in WINDOWS:                candidates.append({"expression": f"{sign}rank({op}({field_id}, {window}))",                                   "field_ids": field_id, "category": category})        for left, right in INTERACTION_PAIRS:            candidates.append({"expression": f"rank(rank({left}) * rank({right}))",                               "field_ids": f"{left}|{right}", "category": "interaction"})    return pd.DataFrame(candidates).drop_duplicates(subset=["expression"]).reset_index(drop=True)candidates = build_candidates(approved)print(f"candidates from {len(approved)} approved fields: {len(candidates)}")candidates.head(10)

## 8. Simulate and collect, writing to disk as you goEach candidate is one real simulation on the platform, and simulations cost quota. The runtherefore starts small, writes every result to the registry immediately, and skips anythingalready recorded — so an interrupted run continues instead of restarting.The metrics are written with the platform's own names (in-sample and out-of-sample) and theout-of-sample Sharpe is also exposed under a project-level name, so the rest of the pipelinedoes not care where a number came from.

In [ ]:
N_TO_SIMULATE = cfg("simulation.max_candidates", 30)SETTINGS = cfg("simulation.settings", {"instrumentType": "EQUITY", "region": REGION,                                        "universe": UNIVERSE, "delay": DELAY})done = set()if REGISTRY_PATH.exists():    done = set(pd.read_csv(REGISTRY_PATH)["expression"].dropna().tolist())records = []for candidate in candidates.to_dict("records"):    if candidate["expression"] in done:        continue    if len(records) >= N_TO_SIMULATE:        break    try:        location = client.simulate(candidate["expression"], SETTINGS)        result = client.poll(location)        alpha = client.get_alpha(result["alpha"]) if result.get("alpha") else {}        record = {            "expression": candidate["expression"],            "field_ids": candidate["field_ids"],            "category": candidate["category"],            "alpha_id": result.get("alpha"),            "is_sharpe": alpha.get("is", {}).get("sharpe"),            "os_sharpe": alpha.get("os", {}).get("sharpe"),            "is_fitness": alpha.get("is", {}).get("fitness"),            "os_fitness": alpha.get("os", {}).get("fitness"),            "turnover": alpha.get("is", {}).get("turnover"),        }    except Exception as exc:        record = {"expression": candidate["expression"], "field_ids": candidate["field_ids"],                  "category": candidate["category"], "alpha_id": None, "error": str(exc)}    records.append(record)    # append immediately: a crash must not lose completed work    pd.DataFrame(records).to_csv(REGISTRY_PATH, mode="a", header=not REGISTRY_PATH.exists(),                                 index=False)registry = pd.read_csv(REGISTRY_PATH)print(f"registry rows: {len(registry)}")registry.tail(5)

## 9. Cheap screenThe first filter only uses in-sample metrics, because those are the ones available for everycandidate without spending more quota. The thresholds come from the configuration file.

In [ ]:
SCREEN = cfg("screen", {"min_sharpe": 1.0, "min_fitness": 1.0, "max_turnover": 0.8})def column(df, name, default=np.nan):    if name in df.columns:        return df[name]    return pd.Series([default] * len(df), index=df.index)is_sharpe = column(registry, "is_sharpe")is_fitness = column(registry, "is_fitness")turnover = column(registry, "turnover")registry["passed_screen"] = (    (is_sharpe >= SCREEN["min_sharpe"])    & (is_fitness >= SCREEN["min_fitness"])    & (turnover <= SCREEN["max_turnover"]))print(f"passed the cheap screen: {int(registry['passed_screen'].sum())} of {len(registry)}")

## 10. Robustness proxyA single simulation returns an in-sample and an out-of-sample segment, not a series ofrolling windows, so the honest proxy for stability is agreement between the two segments.`stability = min(is, os) / max(is, os)` is close to 1 when both segments agree.It is a proxy, not a proof. Where real daily return series exist, rolling windows are thebetter check, and the configuration leaves room for that step.

In [ ]:
def stability(is_value, os_value):    if not np.isfinite(is_value) or not np.isfinite(os_value):        return np.nan    low, high = min(is_value, os_value), max(is_value, os_value)    return low / high if high else np.nanregistry["stability"] = [stability(a, b) for a, b in                         zip(column(registry, "is_sharpe"), column(registry, "os_sharpe"))]registry["validation_sharpe"] = column(registry, "os_sharpe")registry[["expression", "is_sharpe", "os_sharpe", "stability", "passed_screen"]].head()

## 11. Correlation clustering — the part that actually bindsTwo things pass every threshold and still do not belong in the same book: they are toosimilar to each other.The fix is to build a numeric feature vector per candidate from the metrics that describeits behaviour, cluster by correlation, and keep the strongest out-of-sample member of eachcluster. The library then grows in breadth rather than in count.In my own runs this step, not the score threshold, was what limited how much I could add.

In [ ]:
CORR_THRESHOLD = cfg("dedup.corr_threshold", 0.85)FEATURE_COLUMNS = ["is_sharpe", "os_sharpe", "is_fitness", "os_fitness", "turnover"]def feature_vector(row):    values = [row.get(c) for c in FEATURE_COLUMNS]    return np.array([0.0 if v is None or not np.isfinite(v) else float(v) for v in values])def cluster_and_keep(frame):    kept, cluster = [], 0    for _, row in frame.sort_values("validation_sharpe", ascending=False).iterrows():        vector = feature_vector(row)        duplicate = False        for representative in kept:            other = feature_vector(representative)            denominator = np.linalg.norm(vector) * np.linalg.norm(other)            correlation = float(np.dot(vector, other) / denominator) if denominator else 0.0            if correlation > CORR_THRESHOLD:                duplicate = True                break        if not duplicate:            kept.append(row)            cluster += 1    return pd.DataFrame(kept), clustercandidates_passing = registry[registry["passed_screen"]].copy()if len(candidates_passing):    deduped, clusters = cluster_and_keep(candidates_passing)    print(f"{len(candidates_passing)} candidates -> {len(deduped)} representatives "          f"in {clusters} clusters")    deduped.head()else:    print("nothing passed the screen")

## 12. ExportThe export is a flat registry with a stable schema: the expression, the fields it uses, thecategory, the train and validation scores, the turnover, the stability and the duplicationcluster. Downstream analysis reads this file, so its shape matters more than the internals ofthe notebook.

In [ ]:
EXPORT_PATH = ARTIFACTS / cfg("paths.export", "alpha_export.csv")final = registry[registry["passed_screen"]].copy() if "passed_screen" in registry else registry.copy()final["status"] = "PASS"final["created_at"] = pd.Timestamp.now().isoformat()final["train_sharpe"] = column(final, "is_sharpe")final["validation_sharpe"] = column(final, "os_sharpe")final["max_drawdown"] = column(final, "max_drawdown")final["correlation_cluster"] = final.get("cluster", pd.Series([None] * len(final)))export_columns = ["expression", "field_ids", "category", "train_sharpe", "validation_sharpe",                   "turnover", "max_drawdown", "stability", "correlation_cluster", "status",                   "created_at"]final[[c for c in export_columns if c in final.columns]].to_csv(EXPORT_PATH, index=False)print(f"export -> {EXPORT_PATH} ({len(final)} rows)")final.head()

## 13. Experiment logThe question this answers is "have I run this before, and why was that candidate dropped?".Every run records the configuration in force, the library size, the candidate count, thecounts at each gate, and the code version.`git rev-parse HEAD` is used as the code version, so the log ties a result set to a commit.

In [ ]:
def code_version():    try:        return subprocess.check_output(["git", "rev-parse", "HEAD"]).decode().strip()[:8]    except Exception:        return "n/a"experiment = {    "experiment_id": pd.Timestamp.now().strftime("%Y%m%d-%H%M%S"),    "region": REGION, "universe": UNIVERSE, "delay": DELAY,    "fields_in_library": len(fields_df),    "fields_after_coverage_filter": len(fields_filtered),    "fields_approved": len(approved),    "candidates_generated": len(candidates),    "simulated_this_run": len(records),    "registry_rows": len(registry),    "passed_screen": int(registry["passed_screen"].sum()) if "passed_screen" in registry else 0,    "code_version": code_version(),    "config_snapshot": CONFIG,}log_path = ARTIFACTS / cfg("paths.experiment_log", "experiment_log.jsonl")with open(log_path, "a", encoding="utf-8") as handle:    handle.write(json.dumps(experiment, ensure_ascii=False, default=str) + "\n")print(f"experiment logged -> {log_path}")experiment["code_version"]

## 14. Read the platform's verdict instead of re-implementing itThe platform publishes its own checks for every candidate: a set of pass/fail conditionswith a message for each failure. Re-implementing those thresholds is how a pipeline driftsout of sync with the thing it is measuring.So the last step parses the platform's checks and reports what they say. It deliberatelydoes **not** submit anything to an account — submission is a decision a person makes withthe failure reasons in front of them, and a pipeline that submits on its own cannot bereviewed.

In [ ]:
def extract_checks(payload):    """Pull the platform's own checks out of a result payload."""    checks = []    for segment in ("is", "os"):        block = payload.get(segment, {}) if isinstance(payload, dict) else {}        for check in block.get("checks", []) or []:            checks.append({                "segment": segment,                "name": check.get("name"),                "result": check.get("result"),                "limit": check.get("limit"),                "value": check.get("value"),            })    return checksdef summarise(checks):    failed = [c for c in checks if str(c.get("result", "")).upper() == "FAIL"]    if not failed:        return "PASS", []    return "NEEDS IMPROVEMENT", [f"{c['segment']}: {c['name']} (value {c.get('value')}, limit {c.get('limit')})"                                 for c in failed]verdict, reasons = summarise(extract_checks({}))print(f"verdict: {verdict}")for reason in reasons:    print("  -", reason)

## Appendix — running it, and what is not here**Running it**1. Put the platform credentials in the environment (`WORLDQUANT_USERNAME` / `WORLDQUANT_PASSWORD`   or `BRAIN_USERNAME` / `BRAIN_PASSWORD`). Never in the notebook, never in a committed file.2. `pip install -r requirements.txt`3. Copy `factory_config.example.json` to `factory_config.json` and fill in the strategy   choices — operators, windows, interaction pairs, thresholds.4. Run the cells in order. Step 6 writes the review sheet; approve the fields you want and   re-run from there.**What is deliberately not here**- the field list and the expression grammar with my operator and window choices- any per-candidate metric, alpha identifier or submitted portfolio- the failure analysis and the classification of my own results**Known limits**- The rolling-window robustness check is described but not implemented: the platform returns  two segments, not a return series.- The review gate needs a human. It is the step that makes the rest of the pipeline worth  running, and also the step that stops it on its own.